# 03 · Errors, fallbacks and the Router

Tutorial sections 6–7 ([`TUTORIAL.md`](../TUTORIAL.md)).

All models are reached through AWS Bedrock. Run the setup cell first: it loads `.env.local` next to this notebook (AWS profile, region, model choice, `LIVE` switch) and prints those settings. Credentials come from your AWS SSO profile, never from the notebook or the file. With `LIVE` unset or `0`, every call returns a mock response: no AWS access, no cost. See `00_credentials.ipynb` to set up.

`FAST`, `SMART` and `ALT` are the three Bedrock models from `.env.local` (defaults: Claude Haiku 5.5, Claude Opus 5.5, Amazon Nova 2 Lite).

In [1]:
from _env import ALT, FAST, LIVE, MODELS, SMART, mock, status  # loads .env.local; import before litellm

import litellm

litellm.suppress_debug_info = True
status()

import logging

# The fallback runner logs each failed attempt with a full traceback, even when a fallback succeeds.
logging.getLogger("LiteLLM").setLevel(logging.CRITICAL)
logging.getLogger("LiteLLM Router").setLevel(logging.CRITICAL)

env file    : .env.local (found)
mode        : LIVE (real Bedrock calls, real cost)
AWS_PROFILE : set (from .env.local)
AWS_REGION  : us-west-2
MODEL_FAST  : bedrock/us.anthropic.claude-haiku-5-5 (default)
MODEL_SMART : bedrock/us.anthropic.claude-opus-5-5 (default)
MODEL_ALT   : bedrock/us.amazon.nova-2-lite-v1:0 (default)


## 6. Errors, retries and fallbacks

Provider errors arrive as OpenAI exception types. The errors here are always simulated, even with `LIVE=1`, so you can see each type without breaking anything. [Section 6](../TUTORIAL.md#6-errors-retries-and-fallbacks)

In [2]:
messages = [{"role": "user", "content": "hi"}]

for err in (
    litellm.RateLimitError("ThrottlingException", llm_provider="bedrock", model=SMART),
    litellm.ContextWindowExceededError("input is too long", llm_provider="bedrock", model=FAST),
    litellm.AuthenticationError("ExpiredTokenException", llm_provider="bedrock", model=FAST),
):
    try:
        litellm.completion(model=FAST, messages=messages, mock_response=err)
    except litellm.RateLimitError as e:
        print(f"RateLimitError       status={e.status_code}  -> back off and retry")
    except litellm.ContextWindowExceededError as e:
        print(f"ContextWindowExceeded status={e.status_code} -> trim input or use a bigger model")
    except litellm.AuthenticationError as e:
        print(f"AuthenticationError  status={e.status_code}  -> run `aws sso login`, do not retry")

RateLimitError       status=429  -> back off and retry
ContextWindowExceeded status=400 -> trim input or use a bigger model
AuthenticationError  status=401  -> run `aws sso login`, do not retry


The primary model fails (simulated); the fallback is a real call when `LIVE=1`.

A fallback inherits every argument of the original call, including `mock_response`, and only the keys in its own dict replace them. So the fallback sets `mock_response` explicitly: `None` (no mock, a real call) when live, a canned reply offline. Without that, the fallback would raise the same simulated error and every attempt would fail.

In [3]:
resp = litellm.completion(
    model=SMART,
    messages=messages,
    max_tokens=1024,
    mock_response=litellm.ServiceUnavailableError("ServiceUnavailableException", llm_provider="bedrock",
                                                  model=SMART),
    fallbacks=[{"model": ALT, "mock_response": None if LIVE else "hello from the fallback"}],
)
print("primary failed; answer came from", resp.model, "->", repr(resp.choices[0].message.content))

primary failed; answer came from us.amazon.nova-2-lite-v1:0 -> 'Hello! How can I help you today?'


## 7. Router: load balancing and failover

No credentials in `model_list`: every deployment uses the AWS profile from `.env.local`. The two `fast` deployments are the same model in two AWS regions, which is the usual way to spread Bedrock load and survive a regional throttle. With `LIVE=1` the load-balancing demo makes 10 calls instead of 100. [Section 7](../TUTORIAL.md#7-router-load-balancing-and-failover)

In [4]:
from collections import Counter

from litellm import Router

router = Router(
    model_list=[
        {"model_name": "fast", "litellm_params": {
            "model": FAST, "aws_region_name": "us-west-2", **mock("hi")},
         "model_info": {"id": "us-west-2"}},
        {"model_name": "fast", "litellm_params": {
            "model": FAST, "aws_region_name": "us-east-1", **mock("hi")},
         "model_info": {"id": "us-east-1"}},
        {"model_name": "smart", "litellm_params": {"model": SMART, **mock("from smart")}},
        {"model_name": "flaky", "litellm_params": {"model": ALT, **mock("from flaky")}},
    ],
    routing_strategy="simple-shuffle",
    num_retries=1,
    fallbacks=[{"flaky": ["smart"]}],
)

msgs = [{"role": "user", "content": "hi"}]
n = 10 if LIVE else 100
counts = Counter(router.completion(model="fast", messages=msgs, max_tokens=512)._hidden_params["model_id"]
                 for _ in range(n))
print(f"{n} calls to 'fast', by deployment:", dict(counts))

resp = router.completion(model="flaky", messages=msgs, max_tokens=512, mock_testing_fallbacks=True)
print("'flaky' failed; answered by", resp.model, "->", repr(resp.choices[0].message.content))

10 calls to 'fast', by deployment: {'us-east-1': 7, 'us-west-2': 3}
'flaky' failed; answered by us.anthropic.claude-opus-5-5 -> 'Hi there! How can I help you today?'
